# 03. 토큰 기반 청킹

`02_preprocess.ipynb`의 유효 문서를 모델 tokenizer 기준으로 분할합니다. 제목·카테고리 prefix와 특수 토큰을 포함한 최종 입력이 128토큰을 넘지 않는지 검사합니다.

In [1]:
from pathlib import Path
import json
import os
import tempfile

import pandas as pd
from IPython.display import display
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer

def find_data_dir():
    for candidate in [Path.cwd(), Path.cwd() / 'ㅋㅌㅊ', Path.cwd().parent, Path.cwd().parent / 'ㅋㅌㅊ']:
        resolved = candidate.resolve()
        if (resolved / 'output/processed/maple_inven_tips_processed.json').is_file():
            return resolved
    raise FileNotFoundError('02_preprocess.ipynb를 먼저 실행하세요.')

DATA_DIR = find_data_dir()
OUTPUT_ROOT = DATA_DIR / 'output'
SETTINGS_PATH = OUTPUT_ROOT / 'intermediate/pipeline_settings.json'
PROCESSED_PATH = OUTPUT_ROOT / 'processed/maple_inven_tips_processed.json'
CHUNKS_PATH = OUTPUT_ROOT / 'RAG/maple_inven_tips_documents_chunked.json'
settings = json.loads(SETTINGS_PATH.read_text(encoding='utf-8'))
MODEL_NAME = settings['model_name']
CHUNK_TOKENS = settings['chunk_tokens']
OVERLAP_TOKENS = settings['overlap_tokens']
MAX_TOKENS = settings['max_tokens']

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.model_max_length = 10**9  # 길이 측정 중 불필요한 경고만 방지
print('모델 tokenizer:', MODEL_NAME)
print(f'본문 목표={CHUNK_TOKENS}, 중첩={OVERLAP_TOKENS}, 최종 한도={MAX_TOKENS}')

C:\Users\Playdata\Desktop\team3_ 프로젝트1\mle-01-p1-team3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


모델 tokenizer: jhgan/ko-sroberta-multitask
본문 목표=100, 중첩=20, 최종 한도=128


In [2]:
def atomic_write_json(path, value):
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = None
    try:
        with tempfile.NamedTemporaryFile('w', encoding='utf-8', newline='', dir=path.parent, delete=False) as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2)
            stream.write('\n')
            temporary = Path(stream.name)
        os.replace(temporary, path)
    finally:
        if temporary is not None and temporary.exists():
            temporary.unlink()

def count_tokens(text, special_tokens=False):
    return len(tokenizer.encode(text, add_special_tokens=special_tokens, truncation=False))

def truncate_to_tokens(text, token_limit):
    if token_limit < 1:
        return ''
    if count_tokens(text) <= token_limit:
        return text
    low, high = 0, len(text)
    while low < high:
        middle = (low + high + 1) // 2
        if count_tokens(text[:middle]) <= token_limit:
            low = middle
        else:
            high = middle - 1
    return text[:low].rstrip()

def embedding_prefix(record, prefix_tokens):
    raw = f"문서 제목: {record['title']}\n카테고리: {record['category']}"
    return truncate_to_tokens(raw, prefix_tokens)

def build_embedding_text(chunk):
    candidate = f"{chunk['metadata']['embedding_prefix']}\n\n{chunk['page_content']}"
    token_count = count_tokens(candidate, special_tokens=True)
    if token_count > MAX_TOKENS:
        raise ValueError(f"임베딩 입력 토큰 제한 초과: {chunk['id']} ({token_count})")
    return candidate

def chunk_records(records):
    if CHUNK_TOKENS < 1:
        raise ValueError('CHUNK_TOKENS는 1 이상이어야 합니다.')
    if OVERLAP_TOKENS < 0 or OVERLAP_TOKENS >= CHUNK_TOKENS:
        raise ValueError('OVERLAP_TOKENS는 0 이상 CHUNK_TOKENS 미만이어야 합니다.')
    chunks = []
    for record in records:
        prefix_budget = max(1, MAX_TOKENS - CHUNK_TOKENS - 2)
        prefix = embedding_prefix(record, prefix_budget)
        body_budget = min(CHUNK_TOKENS, MAX_TOKENS - count_tokens(prefix) - 2)
        if body_budget < 1:
            raise ValueError(f"본문 토큰 예산이 없습니다: {record['document_id']}")
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=body_budget,
            chunk_overlap=min(OVERLAP_TOKENS, max(0, body_budget - 1)),
            length_function=count_tokens,
            separators=['\n\n', '\n', '. ', '? ', '! ', ' ', ''],
            is_separator_regex=False,
        )
        bodies = [body.strip() for body in splitter.split_text(str(record['content'])) if body.strip()]
        for chunk_index, body in enumerate(bodies):
            chunk_id = f"{record['document_id']}_{chunk_index}"
            metadata = {
                'source': 'guide', 'origin': 'inven_tip', 'source_name': '메이플 인벤 팁과 노하우',
                'document_id': record['document_id'], 'chunk_id': chunk_id, 'chunk_index': chunk_index,
                'article_id': record['article_id'], 'name': record['title'],
                'section_title': record['category'], 'url': record['url'],
                'created_at': record['created_at'], 'views': record['views'], 'likes': record['likes'],
                'text_quality': record['text_quality'], 'embedding_prefix': prefix,
            }
            chunk = {'id': chunk_id, 'page_content': body, 'metadata': metadata}
            build_embedding_text(chunk)
            chunks.append(chunk)
    if len({item['id'] for item in chunks}) != len(chunks):
        raise ValueError('중복 chunk_id가 생성됐습니다.')
    return chunks

In [3]:
processed = json.loads(PROCESSED_PATH.read_text(encoding='utf-8'))
chunks = chunk_records(processed)
atomic_write_json(CHUNKS_PATH, chunks)
token_counts = [count_tokens(build_embedding_text(chunk), special_tokens=True) for chunk in chunks]

display({
    '문서 수': len(processed),
    '청크 수': len(chunks),
    '고유 청크 ID': len({item['id'] for item in chunks}),
    '최소 입력 토큰': min(token_counts),
    '최대 입력 토큰': max(token_counts),
    '128토큰 초과': sum(count > MAX_TOKENS for count in token_counts),
    '저장 파일': str(CHUNKS_PATH),
})
preview = []
for chunk, token_count in zip(chunks[:5], token_counts[:5]):
    preview.append({
        'id': chunk['id'], 'category': chunk['metadata']['section_title'],
        'title': chunk['metadata']['name'], 'tokens': token_count,
        'page_content': chunk['page_content'][:220],
    })
display(pd.DataFrame(preview))

{'문서 수': 299,
 '청크 수': 5506,
 '고유 청크 ID': 5506,
 '최소 입력 토큰': 21,
 '최대 입력 토큰': 128,
 '128토큰 초과': 0,
 '저장 파일': 'C:\\Users\\Playdata\\Desktop\\team3_ 프로젝트1\\mle-01-p1-team3\\ㅋㅌㅊ\\output\\RAG\\maple_inven_tips_documents_chunked.json'}

,id,category,title,tokens,page_content
0,inven_tip_48082_0,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,114,메이플의 수많은 시스템들 중 뉴비가 접하기 쉽지 않거나 정보가 파편화 되어있어 알기...
1,inven_tip_48082_1,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,118,틀린 부분이 있다면 지적 부탁드립니다.\n제목 : 쿨타임 시스템\n부제 : 쿨뚝과 ...
2,inven_tip_48082_2,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,113,"스킬의 쿨타임이 줄어들면 그만큼 자주 사용할 수 있게 된다는 뜻이구요,\n좁게 보면..."
3,inven_tip_48082_3,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,114,"메르세데스 200레벨 기준 5%가 감소하며, 250레벨을 달성해야 6%가 감소합니다..."
4,inven_tip_48082_4,실험,뉴비에서 메잘알까지 1편 - 쿨타임 시스템,94,> 쿨타임 5% 감소가 큰가요? 별로 안커보이는데\n> 5%는 1200초(20분) ...
